In [13]:
## 基本工具定义
from langchain.tools import tool

# 装饰器 -- 下面单引号包裹的内容(原来是解释函数的)变为工具的描述，帮助模型理解
# 工具名称(即函数名称)选择用蛇形命名法(snake_case)，即字母、数字字符、下划线和连字符
@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.
    
    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [14]:
## 自定义工具属性
### 自定义工具名称
@tool("web_search")     # 自定义名称
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)      # 输出 web_search

web_search


In [15]:
### 自定义工具描述(类似 skills 的触发规则)
@tool("calculator", description="Performs arithmetic calculators. Use this for math prol=blems.")
def calc(expression: str) -> str:
    """Evaluate mathmatical expressions."""
    return str(eval(expression))

**① 用户输入**  
你在 Claude Code 对话框发送：*"上海的天气怎么样？用摄氏度显示"*

**② 大模型读取工具说明书**  
Claude(模型) 分析意图后，查阅系统提示词中由 `@tool` 生成的 `WeatherInput` 类说明书（包含 `location` 必填、`units` 默认 `celsius` 等描述）。

**③ 大模型生成参数 JSON**  
Claude(模型) 根据说明书输出调用参数（省略了有默认值的字段）：  
`{"location": "上海"}`

**④ Agent 框架校验并实例化**  
后台捕获 JSON，用 `WeatherInput` 类验证，自动补全缺失的默认值：  
`WeatherInput(location="上海", units="celsius", include_forecast=False)`

**⑤ 框架调用你的函数**  
框架将 Pydantic 对象拆解，实际执行：  
`get_weather(location="上海", units="celsius", include_forecast=False)`

**⑥ 函数执行业务逻辑**  
- 判断 `units == "celsius"`，设 `temp = 22`  
- 拼接字符串（此时 `.upper()` 正确执行，取 `"celsius"[0].upper()` 得到 `"C"`）  
- `include_forecast` 为 `False`，不追加预报内容  
- 结果暂存为：`"Current weather in 上海: 22 degrees C"`

**⑦ 函数返回原始结果**  
`get_weather` 将上述字符串返回给 Agent 框架。

**⑧ 大模型翻译并回复用户**  
框架将字符串传给 Claude，Claude 重新组织成自然语言输出到你的聊天框：  
*"上海当前气温为 22 摄氏度。"*

In [16]:
## 高级模式自定义
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):          # 继承 创建数据模型的基类 BaseModel(让定义的 WeatherInput 类具备自动校验参数类型、必填项的能力)
    """Input for weather queries."""
    # location 为 str 类型的属性，Field 函数 为模型字段添加额外元数据，这里添加 description，让描述可以被大模型读取
    location: str = Field(
        description="City name or coordinates"  # 给大模型看的提示语，告诉模型“请在这里填城市名或坐标”
    )
    # Literal:精确限定值范围，这里明确限制 单位(units) 为摄氏度(celsius)或者华氏度(fahrenheit)
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius", 
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecase"
    )

# 这个装饰器告诉底层的 AI 框架：“下面的 get_weather 函数是一个可被大模型调用的工具，传入参数的格式请严格按照 WeatherInput 这个类的定义来解析
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) ->str:   # 参数列表要与 WeatherInput 保持一致
    """Get current weather and optional forecase."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:    # 如果用户要求包含预报，就在结果末尾追加接下来五天的天气（这里硬编码为晴朗）
        result += "\nNext 5 days: Sunny"
    return result

## 保留参数名称 -- config 和 runtime 参数不能用于工具参数，否则会导致运行错误

In [19]:
# 访问上下文
## 1.短期记忆(状态): runtime: ToolRuntime
### 1.1 访问状态：runtime.state

# ToolRuntime类 是运行时上下文对象，提供对当前状态（state）、配置、回调等的访问，在工具函数内部可以通过参数注入获取
from langchain.tools import tool, ToolRuntime
# HumanMessage 类，是 LangChain 消息体系中的一种，表示来自用户（人类）的消息。在消息列表中，可以通过类型判断来区分用户消息、AI 消息或系统消息
from langchain.messages import HumanMessage

# 工具 1：从标准消息历史中提取最近的用户输入
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    # runtime.state 是工具执行时的状态字典，通常包含对话历史、用户偏好、上下文数据等；"messages" 是 LangChain 的标准键，存储了当前对话的所有消息列表（按时间顺序，最早在前）
    messages = runtime.state["messages"]
    
    # Find the last human message
    for message in reversed(messages):  # reversed 翻转，先取最近的信息
        if isinstance(message, HumanMessage):   # 有实例返回，即有最近的信息，直接返回其内容（一般为 str)
            return message.content
        
    return "No user messages found"

# Access custom state fields
# 工具 2：从自定义状态字段（用户偏好）中读取指定项
@tool
def get_user_preference(
    pref_name: str, 
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    # 从 runtime.state 中获取键为 "user_preferences" 的值，如果该键不存在则返回空字典 {}；"user_preferences" 是一个自定义状态字段，不是 LangChain 内置的，但开发者可以在外部将其注入到 state 中（例如通过 AgentExecutor 的 input 或 chain 的配置）
    preferences = runtime.state.get("user_preferences", {})
    # 从 preferences 字典中根据传入的 pref_name 键获取对应的值，不存在返回 str "Not set"
    return preferences.get(pref_name, "Not set")

**状态更新代码的流程案例**：

```text
[用户: "我叫小明"]
       ↓
[LLM → AIMessage(tool_calls=[set_user_name])]
       ↓
[LangGraph 调用工具]
   → new_name="小明"
   → runtime.tool_call_id="call_abc123"
       ↓
[工具返回 Command(update={...})]
       ↓
[LangGraph 原子性合并状态]
   → messages 追加 ToolMessage
   → user_name = "小明"
       ↓
[LLM 读取更新后状态 → "好的小明"]
```

In [20]:
### 1.2 更新状态：Command

# 导入 AgentState 类。这是 LangChain 代理（Agent）的基础状态类，通常作为构建自定义状态的父类，包含了 messages（消息列表）等核心字段
from langchain.agents import AgentState
# 导入 ToolMessage。这是 LangChain 消息体系中的一种特殊消息类型，专门用于返回工具（Tool）的执行结果。它必须包含一个 tool_call_id，用于关联触发这次调用的那条 ToolCall 请求
from langchain.messages import ToolMessage
# 用于定义可被代理(Agent)调用的工具
from langchain.tools import ToolRuntime, tool
# 关键：Command 是 LangGraph 中用于控制状态流转的特殊对象。当工具返回 Command 时，LangGraph 不会仅仅把返回值当作普通内容，而是会解析 Command 内部的指令（如 update、goto）来更新图状态（State）或跳转节点现
from langgraph.types import Command

class CustomState(AgentState):      # 继承弗雷 AgentState
    # 新增字段 user_name，意味着在整个对话图（Graph）的执行过程中，state 字典里除了标准的 messages 外，还会持久化保存一个 user_name 字段，且可以在不同节点间传递
    user_name: str                  
    
@tool
# runtime: ToolRuntime[None, CustomState]：这是泛型类型标注。第一个泛型参数 None 表示上下文（Context）为空；第二个泛型参数 CustomState 明确告诉类型检查器和框架，这个工具操作的状态是 CustomState 类型
# 返回值标注为 Command，表示该工具将通过返回指令来修改状态，而不是直接返回普通字符串
def set_user_name(new_name: str, runtime: ToolRuntime[None, CustomState]) -> Command:
    """Set the user's name in the conversation state."""
    # 构造并返回一个 Command 对象，核心在于 update 字典，它定义了要对全局状态执行的更新操作
    return Command(
        update={
            "user_name": new_name, 
            # LangGraph 默认对列表类型的更新采用追加或合并策略，具体取决于 reducer 函数，但标准 messages 字段配置为追加
            "messages": [
                ToolMessage(
                    content=f"User name set to {new_name}.",
                    # 重点：runtime.tool_call_id 是从运行时上下文中获取的本次工具调用的唯一 ID。LangGraph 要求 ToolMessage 必须对应一个先前存在的 ToolCall，通过这个 ID 将结果与请求配对，否则图执行会报错
                    tool_call_id=runtime.tool_call_id, 
                )
            ],
        }
    )

In [23]:
## 2. 上下文

# Python 标准库的 dataclass 装饰器，自动生成 __init__、__repr__ 等方法，用于快速定义类
from dataclasses import dataclass
# 加载环境变量，避免硬编码 API_KEY 暴露
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
# 导入高级 Agent 工厂函数。它封装了 LangGraph 的图构建、工具绑定、系统提示词注入等 boilerplate 代码
from langchain.agents import create_agent
# ToolRuntime：运行时上下文注入器，使工具能访问 Context 和 State 字段
from langchain.tools import tool, ToolRuntime

# 模拟的用户数据库字典。注意：这是全局只读数据源，不是 Graph State。实际项目中应替换为数据库查询或 API 调用
USER_DATABSE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000, 
        "email": "alice@example.com"
    }, 
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

# 定义上下文数据结构。仅包含 user_id 字段，标识当前会话所属用户。
# 使用 @dataclass 后，可直接通过 UserContext(user_id="user123") 构造实例，无需手写 __init__
@dataclass
class UserContext:
    user_id: str
    
@tool
# 定义工具函数。关键特征：没有 LLM 传入的业务参数，仅依赖 runtime。
# ToolRuntime[UserContext]：明确声明需要 UserContext 类型的上下文。LLM 调用此工具时无需传参，框架自动注入 context。
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    # 从运行时上下文中提取 user_id
    # 安全性体现：用户 ID 来自应用层注入的 Context，而非 LLM 生成的参数，彻底避免 LLM 幻觉或提示注入导致的越权访问
    user_id = runtime.context.user_id

    # 查询模拟数据库并格式化返回字符串
    # 返回值是普通 str，框架会自动将其包装为 ToolMessage 追加到消息列表。因为不需要修改 State，所以无需返回 Command
    if user_id in USER_DATABSE:
        user = USER_DATABSE[user_id]
        return f"Account holder: {user['name']}\nType: {user['account_type']}\nBalance: ${user['balance']}"
    return "User not found"

# 加载环境变量，必须在模型定义前面
load_dotenv()

# 初始化 LLM 实例
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0.7
)

# 创建 Agent 实例
agent = create_agent(
    model, 
    tools=[get_account_info],       # 绑定可用工具 **列表**（可多个）
    context_schema=UserContext,     # 声明式契约，告知框架此 Agent 需要 UserContext，后续 invoke 时必须传入匹配的 context
    system_prompt="You are a financial assistant."
)

result = agent.invoke(              # invoke 调用
    # 第一个参数：初始 State（仅需提供用户消息）
    {"messages": [{"role": "user", "content": "What's my current balance?"}]}, 
    # 外部注入上下文。此值在整个图执行期间只读可用，不会被任何节点修改或持久化到 State 中
    context=UserContext(user_id="user123")      # Alice Johnson
)

# 格式化查看完整对话流
for msg in result["messages"]:
    role = type(msg).__name__           # HumanMessage / AIMessage / ToolMessage
    print(f"[{role}] {msg.content}")

[HumanMessage] What's my current balance?
[AIMessage] Let me check your account information!
[ToolMessage] Account holder: Alice Johnson
Type: Premium
Balance: $5000
[AIMessage] Here's your account information:

- **Account Holder:** Alice Johnson
- **Account Type:** Premium
- **Current Balance:** **$5,000.00**

Is there anything else I can help you with? 😊
